# Day 071 — Solution: Visual Search Engine

In [ ]:
_SEARCH_SRC = '"""visual_search.py — Day 071: Vision + RAG.\n\nContent-based image retrieval: describe images with a vision LLM,\nembed descriptions with a text model, search by text or image query.\n\nSetup (real usage):\n    ollama pull llava              # vision model\n    ollama pull nomic-embed-text   # embedding model\n\nUsage:\n    from visual_search import ImageSearchEngine\n    from PIL import Image\n\n    # Offline testing — inject mocks\n    mock_describe = lambda b64, p: "a red car on a road"\n    mock_embed    = lambda text: [0.1, 0.9, 0.2, 0.4]\n    engine = ImageSearchEngine(describe_fn=mock_describe, embed_fn=mock_embed)\n    img = Image.new("RGB", (64, 64), "red")\n    engine.add_image("img1", img, metadata={"label": "car"})\n    results = engine.search("car", n=1)\n    print(results[0]["id"], results[0]["score"])\n"""\nimport base64\nimport io\nimport math\nimport numpy as np\nfrom PIL import Image\nfrom typing import Optional, Callable\n\n_SEARCH_PROMPT = (\n    "Describe this image in detail for use in a semantic search index. "\n    "Include: main subjects, colors, textures, setting, and any visible text. "\n    "Write one concise paragraph of 2-3 sentences."\n)\n\n\ndef image_to_base64(img: Image.Image, format: str = "PNG") -> str:\n    """Convert PIL Image to base64 string."""\n    buf = io.BytesIO()\n    out = img\n    if format.upper() in ("JPEG", "JPG") and img.mode in ("RGBA", "P"):\n        out = img.convert("RGB")\n    out.save(buf, format=format)\n    return base64.b64encode(buf.getvalue()).decode()\n\n\ndef cosine_similarity(a, b) -> float:\n    """Cosine similarity between two vectors. Returns 0.0 if either is zero."""\n    va = np.array(a, dtype=np.float32)\n    vb = np.array(b, dtype=np.float32)\n    denom = float(np.linalg.norm(va) * np.linalg.norm(vb))\n    if denom == 0.0:\n        return 0.0\n    return float(np.dot(va, vb) / denom)\n\n\nclass ImageIndex:\n    """In-memory image search index backed by cosine similarity."""\n\n    def __init__(self) -> None:\n        self._items: list = []\n\n    def add(self, image_id: str, description: str, embedding,\n            metadata: Optional[dict] = None) -> None:\n        """Add an image to the index."""\n        self._items.append({\n            "id":          image_id,\n            "description": description,\n            "embedding":   np.array(embedding, dtype=np.float32),\n            "metadata":    metadata or {},\n        })\n\n    def search(self, query_embedding, n: int = 5) -> list:\n        """Return top-n results sorted by cosine similarity (descending).\n\n        Each result dict has keys: id, description, score, metadata.\n        """\n        if not self._items:\n            return []\n        q = np.array(query_embedding, dtype=np.float32)\n        scored = [\n            (cosine_similarity(q, item["embedding"]), item)\n            for item in self._items\n        ]\n        scored.sort(key=lambda x: -x[0])\n        top = scored[:min(n, len(scored))]\n        return [\n            {\n                "id":          item["id"],\n                "description": item["description"],\n                "score":       float(score),\n                "metadata":    item["metadata"],\n            }\n            for score, item in top\n        ]\n\n    def __len__(self) -> int:\n        return len(self._items)\n\n\ndef describe_image_for_search(img: Image.Image,\n                               describe_fn: Optional[Callable] = None) -> str:\n    """Generate a text description of an image for search indexing.\n\n    Args:\n        img:         PIL Image\n        describe_fn: callable(img_b64, prompt) -> str for testing\n    Returns:\n        Text description string\n    """\n    img_b64 = image_to_base64(img)\n    if describe_fn is not None:\n        return describe_fn(img_b64, _SEARCH_PROMPT)\n    import ollama\n    resp = ollama.chat(\n        model="llava",\n        messages=[{\n            "role":    "user",\n            "content": _SEARCH_PROMPT,\n            "images":  [img_b64],\n        }],\n    )\n    return resp["message"]["content"].strip()\n\n\ndef embed_text(text: str, embed_fn: Optional[Callable] = None) -> list:\n    """Embed text to a float vector.\n\n    Args:\n        text:     Input string\n        embed_fn: callable(text) -> list[float] for testing\n    Returns:\n        list of float (embedding vector)\n    """\n    if embed_fn is not None:\n        return embed_fn(text)\n    import ollama\n    resp = ollama.embeddings(model="nomic-embed-text", prompt=text)\n    return resp["embedding"]\n\n\ndef index_images(images_with_ids: list,\n                 describe_fn: Optional[Callable] = None,\n                 embed_fn: Optional[Callable] = None) -> "ImageIndex":\n    """Describe, embed, and index a batch of images.\n\n    Args:\n        images_with_ids: list of (image_id, img, metadata) tuples\n        describe_fn:     callable(img_b64, prompt) -> str for testing\n        embed_fn:        callable(text) -> list[float] for testing\n    Returns:\n        Populated ImageIndex\n    """\n    index = ImageIndex()\n    for image_id, img, metadata in images_with_ids:\n        desc = describe_image_for_search(img, describe_fn=describe_fn)\n        emb  = embed_text(desc, embed_fn=embed_fn)\n        index.add(image_id, desc, emb, metadata or {})\n    return index\n\n\ndef search_by_text(query: str, index: "ImageIndex",\n                   embed_fn: Optional[Callable] = None,\n                   n: int = 5) -> list:\n    """Search the index by a text query.\n\n    Args:\n        query:    Text search query\n        index:    Populated ImageIndex\n        embed_fn: callable(text) -> list[float] for testing\n        n:        Maximum number of results\n    Returns:\n        list of result dicts (id, description, score, metadata)\n    """\n    q_emb = embed_text(query, embed_fn=embed_fn)\n    return index.search(q_emb, n=n)\n\n\ndef search_by_image(img: Image.Image, index: "ImageIndex",\n                    describe_fn: Optional[Callable] = None,\n                    embed_fn: Optional[Callable] = None,\n                    n: int = 5) -> list:\n    """Search the index using an image as the query.\n\n    Args:\n        img:         Query PIL Image\n        index:       Populated ImageIndex\n        describe_fn: callable(img_b64, prompt) -> str for testing\n        embed_fn:    callable(text) -> list[float] for testing\n        n:           Maximum number of results\n    Returns:\n        list of result dicts (id, description, score, metadata)\n    """\n    desc  = describe_image_for_search(img, describe_fn=describe_fn)\n    q_emb = embed_text(desc, embed_fn=embed_fn)\n    return index.search(q_emb, n=n)\n\n\nclass ImageSearchEngine:\n    """Content-based image search engine.\n\n    Inject describe_fn and embed_fn for testing without Ollama::\n\n        engine = ImageSearchEngine(\n            describe_fn=lambda b64, p: "a sunny beach",\n            embed_fn=lambda t: [0.5, 0.3, 0.8],\n        )\n    """\n\n    def __init__(self, describe_fn: Optional[Callable] = None,\n                 embed_fn: Optional[Callable] = None) -> None:\n        self._describe_fn = describe_fn\n        self._embed_fn    = embed_fn\n        self._index       = ImageIndex()\n\n    def add_image(self, image_id: str, img: Image.Image,\n                  metadata: Optional[dict] = None) -> str:\n        """Index one image. Returns the generated description."""\n        desc = describe_image_for_search(img, describe_fn=self._describe_fn)\n        emb  = embed_text(desc, embed_fn=self._embed_fn)\n        self._index.add(image_id, desc, emb, metadata or {})\n        return desc\n\n    def add_batch(self, images_with_ids: list) -> list:\n        """Index a batch of (image_id, img, metadata) tuples.\n\n        Returns list of generated description strings.\n        """\n        return [\n            self.add_image(image_id, img, metadata)\n            for image_id, img, metadata in images_with_ids\n        ]\n\n    def search(self, query: str, n: int = 5) -> list:\n        """Search by text query. Returns list of result dicts."""\n        return search_by_text(query, self._index,\n                              embed_fn=self._embed_fn, n=n)\n\n    def search_by_image(self, img: Image.Image, n: int = 5) -> list:\n        """Search by image query. Returns list of result dicts."""\n        return search_by_image(img, self._index,\n                               describe_fn=self._describe_fn,\n                               embed_fn=self._embed_fn, n=n)\n\n    def __len__(self) -> int:\n        return len(self._index)\n'
from pathlib import Path
Path('visual_search.py').write_text(_SEARCH_SRC, encoding='utf-8')
print('visual_search.py written.')

In [ ]:
import hashlib
from PIL import Image
from visual_search import (
    cosine_similarity, ImageIndex,
    describe_image_for_search, embed_text,
    index_images, search_by_text, search_by_image,
    ImageSearchEngine,
)

def _mock_desc(b64, p):
    h = int(hashlib.md5(b64.encode()).hexdigest()[:4], 16)
    return ['a red apple', 'a blue ocean', 'a green forest'][h % 3]

def _mock_emb(t):
    h = int(hashlib.md5(t.encode()).hexdigest()[:8], 16)
    return [((h >> (i*8)) & 0xff) / 128.0 - 1.0 for i in range(4)]

# 1. cosine_similarity
assert abs(cosine_similarity([1,0], [1,0]) - 1.0) < 1e-5
assert abs(cosine_similarity([1,0], [0,1]))       < 1e-5
assert cosine_similarity([0,0], [1,2]) == 0.0
print("\u2705 cosine_similarity correct")

# 2. ImageIndex
idx = ImageIndex()
idx.add('a', 'red car', [1.0, 0.0])
idx.add('b', 'blue sky', [0.0, 1.0])
assert len(idx) == 2
results = idx.search([1.0, 0.0], n=2)
assert results[0]['id'] == 'a' and results[0]['score'] > results[1]['score']
print("\u2705 ImageIndex correct")

# 3. describe_image_for_search
img = Image.new('RGB', (16,16), 'red')
desc = describe_image_for_search(img, describe_fn=_mock_desc)
assert isinstance(desc, str) and len(desc) > 0
print("\u2705 describe_image_for_search correct")

# 4. embed_text
emb = embed_text('a red car', embed_fn=_mock_emb)
assert isinstance(emb, list) and len(emb) == 4
print("\u2705 embed_text correct")

# 5. index_images
imgs = [('i1', Image.new('RGB',(16,16),(220,50,50)), {'tag':'red'}),
        ('i2', Image.new('RGB',(16,16),(50,100,220)), {'tag':'blue'})]
built_idx = index_images(imgs, describe_fn=_mock_desc, embed_fn=_mock_emb)
assert len(built_idx) == 2
print("\u2705 index_images correct")

# 6. search_by_text
r = search_by_text('apple', built_idx, embed_fn=_mock_emb, n=2)
assert len(r) <= 2 and all('score' in x for x in r)
print("\u2705 search_by_text correct")

# 7. search_by_image
q = Image.new('RGB', (16,16), 'blue')
r2 = search_by_image(q, built_idx, describe_fn=_mock_desc, embed_fn=_mock_emb, n=1)
assert len(r2) == 1 and 'id' in r2[0]
print("\u2705 search_by_image correct")

# 8. ImageSearchEngine
engine = ImageSearchEngine(describe_fn=_mock_desc, embed_fn=_mock_emb)
for iid, img_x, meta in imgs:
    engine.add_image(iid, img_x, meta)
assert len(engine) == 2
sr = engine.search('blue ocean', n=1)
assert len(sr) == 1 and sr[0]['score'] >= -1.0
print("\u2705 ImageSearchEngine correct")

print("\nVisual Search Engine complete!")
